# Лабораторная работа №4 — Qiskit (упражнения из методичка.pdf)

Этот ноутбук генерирует артефакты (схемы, гистограммы, данные) для отчета `lab-1IBM.tex`.

In [20]:
def _fig_to_array(fig) -> np.ndarray:
    """Convert a matplotlib figure to a numpy array (RGB)."""
    fig.canvas.draw()
    buf = np.asarray(fig.canvas.buffer_rgba())
    return buf[..., :3].copy()

In [21]:
# Exercise 1 — гистограммы для задачи 1.3
qc_h = QuantumCircuit(1, 1)
qc_h.h(0)
qc_h.measure(0, 0)

shots_list = [1, 2, 8, 32, 64, 128, 512, 1024, 8192]
counts_series = [run_counts(qc_h, s) for s in shots_list]

ncols = 3
nrows = int(np.ceil(len(shots_list) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3.2*nrows), dpi=200)
axes = axes.flatten()
for ax, cnt, s in zip(axes, counts_series, shots_list):
    plot_histogram(cnt, ax=ax, bar_labels=False, title=f'shots={s}')
    ax.set_ylim(0, max(cnt.values()) if cnt else 1)
for ax in axes[len(shots_list):]:
    ax.axis('off')
fig.tight_layout()
fig.savefig(FIG_DIR / '1.3.png', bbox_inches='tight')
plt.close(fig)

print('Saved: 1.3.png')

Saved: 1.3.png


In [22]:
# Exercise 1 — generate remaining figures (1.2, 1.4–1.6)
from math import pi

# 1.2 — two-qubit: |0>, |1>, measure both
qc_12 = QuantumCircuit(2, 2)
qc_12.x(1)  # set second qubit to |1>
qc_12.measure([0, 1], [0, 1])
save_circuit_with_histogram(qc_12, FIG_DIR / '1.2.png', shots=SHOTS_DEFAULT, hist_title='Counts (shots=4096)')

# 1.4 — reproduce two Bell-like variants with histograms
qc_14a = QuantumCircuit(2, 2)
qc_14a.h(0)
qc_14a.cx(0, 1)
qc_14a.measure([0, 1], [0, 1])

qc_14b = QuantumCircuit(2, 2)
qc_14b.h(1)
qc_14b.cx(1, 0)
qc_14b.measure([0, 1], [0, 1])

save_circuit_hist_grid([qc_14a, qc_14b], FIG_DIR / '1.4.png', ['1.4a', '1.4b'], shots=SHOTS_DEFAULT)

# 1.5 — two additional two-qubit variants with histograms
qc_15a = QuantumCircuit(2, 2)
qc_15a.h(1)
qc_15a.cx(0, 1)
qc_15a.h(0)
qc_15a.measure([0, 1], [0, 1])

qc_15b = QuantumCircuit(2, 2)
qc_15b.h(1)
qc_15b.cx(1, 0)
qc_15b.h(0)
qc_15b.cx(0, 1)
qc_15b.measure([0, 1], [0, 1])

save_circuit_hist_grid([qc_15a, qc_15b], FIG_DIR / '1.5.png', ['1.5a', '1.5b'], shots=SHOTS_DEFAULT)

# 1.6 — Bloch/Q-sphere snapshots for different single-qubit states
states = {}
states['1.6.1.png'] = Statevector.from_label('0')
states['1.6.2.png'] = Statevector.from_label('1')

qc_tmp = QuantumCircuit(1)
qc_tmp.h(0)
states['1.6.3.png'] = Statevector.from_instruction(qc_tmp)

qc_tmp = QuantumCircuit(1)
qc_tmp.x(0)
qc_tmp.h(0)
states['1.6.4.png'] = Statevector.from_instruction(qc_tmp)

qc_tmp = QuantumCircuit(1)
qc_tmp.rx(pi/3, 0)
states['1.6.5.png'] = Statevector.from_instruction(qc_tmp)

qc_tmp = QuantumCircuit(1)
qc_tmp.rx(pi/3, 0)
qc_tmp.x(0)
states['1.6.6.png'] = Statevector.from_instruction(qc_tmp)

for fname, sv in states.items():
    save_state_bloch_and_qsphere(sv, FIG_DIR / fname, main_title=f'State {fname}')

print('Saved Exercise 1 images: 1.2.png, 1.3.png, 1.4.png, 1.5.png, 1.6.1–1.6.6.png')

Saved Exercise 1 images: 1.2.png, 1.3.png, 1.4.png, 1.5.png, 1.6.1–1.6.6.png


In [23]:
# Exercise 2 — tasks 2.1–2.14 for вариант 15
from math import pi, asin, sqrt

VARIANT = 15
p1 = 0.80
p0 = 0.20
a_mag = sqrt(p0)
b_mag = sqrt(p1)
theta_rot = 2 * asin(b_mag)  # rotation angle for Ry/Rx/U
shots = SHOTS_DEFAULT

print(f'Variant {VARIANT}: P(|0>)={p0:.2f}, P(|1>)={p1:.2f}, theta={theta_rot:.4f} rad')

# 2.1 — H|0>
qc_21 = QuantumCircuit(1, 1)
qc_21.h(0)
qc_21.measure(0, 0)
save_circuit_with_histogram(qc_21, FIG_DIR / '2.1.png', shots=shots, hist_title=f'shots={shots}')

# 2.2 — two ways to obtain (|0>-|1>)/sqrt(2)
qc_221 = QuantumCircuit(1, 1)
qc_221.x(0)
qc_221.h(0)
qc_221.measure(0, 0)
save_circuit_with_histogram(qc_221, FIG_DIR / '2.2.1.png', shots=shots, hist_title=f'shots={shots}')

qc_222 = QuantumCircuit(1, 1)
qc_222.h(0)
qc_222.z(0)
qc_222.measure(0, 0)
save_circuit_with_histogram(qc_222, FIG_DIR / '2.2.2.png', shots=shots, hist_title=f'shots={shots}')

# 2.3 — (-|0> + |1>)/sqrt(2)
qc_23 = QuantumCircuit(1, 1)
qc_23.x(0)
qc_23.h(0)
qc_23.z(0)
qc_23.measure(0, 0)
save_circuit_with_histogram(qc_23, FIG_DIR / '2.3.png', shots=shots, hist_title=f'shots={shots}')

# 2.4 — Rx prepares a|0> + b|1> (probabilities from variant)
qc_24 = QuantumCircuit(1, 1)
qc_24.rx(theta_rot, 0)
qc_24.measure(0, 0)
save_circuit_with_histogram(qc_24, FIG_DIR / '2.4.png', shots=shots, hist_title=f'shots={shots}')

# 2.5 — Ry prepares a|0> + b|1>
qc_25 = QuantumCircuit(1, 1)
qc_25.ry(theta_rot, 0)
qc_25.measure(0, 0)
save_circuit_with_histogram(qc_25, FIG_DIR / '2.5.png', shots=shots, hist_title=f'shots={shots}')

# 2.6 — U(theta,0,0) prepares same state
qc_26 = QuantumCircuit(1, 1)
qc_26.u(theta_rot, 0.0, 0.0, 0)
qc_26.measure(0, 0)
save_circuit_with_histogram(qc_26, FIG_DIR / '2.6.png', shots=shots, hist_title=f'shots={shots}')

# 2.7 — Rx then Sdg to introduce negative sign on |1>
qc_27 = QuantumCircuit(1, 1)
qc_27.rx(theta_rot, 0)
qc_27.sdg(0)
qc_27.measure(0, 0)
save_circuit_with_histogram(qc_27, FIG_DIR / '2.7.png', shots=shots, hist_title=f'shots={shots}')

# 2.8 — Ry then Z to flip sign
qc_28 = QuantumCircuit(1, 1)
qc_28.ry(theta_rot, 0)
qc_28.z(0)
qc_28.measure(0, 0)
save_circuit_with_histogram(qc_28, FIG_DIR / '2.8.png', shots=shots, hist_title=f'shots={shots}')

# 2.9 — U then Z
qc_29 = QuantumCircuit(1, 1)
qc_29.u(theta_rot, 0.0, 0.0, 0)
qc_29.z(0)
qc_29.measure(0, 0)
save_circuit_with_histogram(qc_29, FIG_DIR / '2.9.png', shots=shots, hist_title=f'shots={shots}')

# 2.10 — rotations Rz-Ry-Rz to prepare target state
qc_210 = QuantumCircuit(1, 1)
qc_210.rz(-pi/2, 0)
qc_210.ry(theta_rot, 0)
qc_210.rz(pi/2, 0)
qc_210.measure(0, 0)
save_circuit_with_histogram(qc_210, FIG_DIR / '2.10.png', shots=shots, hist_title=f'shots={shots}')

# 2.11 — Rx then H (per figure 20)
qc_211 = QuantumCircuit(1, 1)
qc_211.rx(theta_rot, 0)
qc_211.h(0)
qc_211.measure(0, 0)
save_circuit_with_histogram(qc_211, FIG_DIR / '2.11.png', shots=shots, hist_title=f'shots={shots}')

# 2.12 — Rx then two H gates (figure 21)
qc_212 = QuantumCircuit(1, 1)
qc_212.rx(theta_rot, 0)
qc_212.h(0)
qc_212.h(0)
qc_212.measure(0, 0)
save_circuit_with_histogram(qc_212, FIG_DIR / '2.12.png', shots=shots, hist_title=f'shots={shots}')

# 2.13 — three circuits from figure 22
qc_213_1 = QuantumCircuit(1, 1)
qc_213_1.h(0)
qc_213_1.measure(0, 0)
save_circuit_with_histogram(qc_213_1, FIG_DIR / '2.13.1.png', shots=shots, hist_title=f'shots={shots}')

qc_213_2 = QuantumCircuit(1, 1)
qc_213_2.x(0)
qc_213_2.h(0)
qc_213_2.measure(0, 0)
save_circuit_with_histogram(qc_213_2, FIG_DIR / '2.13.2.png', shots=shots, hist_title=f'shots={shots}')

qc_213_3 = QuantumCircuit(1, 1)
qc_213_3.z(0)
qc_213_3.h(0)
qc_213_3.measure(0, 0)
save_circuit_with_histogram(qc_213_3, FIG_DIR / '2.13.3.png', shots=shots, hist_title=f'shots={shots}')

# 2.14 — two-qubit schemes from figure 23
qc_214_1 = QuantumCircuit(2, 2)
qc_214_1.h(0)
qc_214_1.h(1)
qc_214_1.measure([0, 1], [0, 1])
save_circuit_with_histogram(qc_214_1, FIG_DIR / '2.14.1.png', shots=shots, hist_title=f'shots={shots}')

qc_214_2 = QuantumCircuit(2, 2)
qc_214_2.h(0)
qc_214_2.cx(0, 1)
qc_214_2.measure([0, 1], [0, 1])
save_circuit_with_histogram(qc_214_2, FIG_DIR / '2.14.2.png', shots=shots, hist_title=f'shots={shots}')

print('Saved Exercise 2 images for variant 15.')

Variant 15: P(|0>)=0.20, P(|1>)=0.80, theta=2.2143 rad
Saved Exercise 2 images for variant 15.
Saved Exercise 2 images for variant 15.
